In [2]:
# All imports + the LLM client used in this notebook
from langchain_openai import ChatOpenAI
import os
from langchain_core.messages import HumanMessage,SystemMessage,AIMessage
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate
from langchain_core.output_parsers import StrOutputParser
# RunnableParallel = run several chains at the same time on the same input
from langchain_core.runnables import RunnableSequence,RunnableParallel

llm = ChatOpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    # api_key=os.getenv("OPENAI_API_KEY"),
    model="nvidia/nemotron-3.5-lightning-30b-a3b",
)

In [3]:
#Task-1[Prompt]
# Common first step: answer / explain the user's topic
prompt_template = ChatPromptTemplate.from_messages([
    ("system","you are a helpful assistant"),
    ("user","{input}")
])

In [4]:
#Task-2[LLM]
llm = ChatOpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    # api_key=os.getenv("OPENAI_API_KEY"),
    model="nvidia/nemotron-3.5-lightning-30b-a3b",
)

In [5]:
#Task-3[Str parser]
str_parser = StrOutputParser()

In [6]:
#Task-4[Custom Runnable]
# string -> {"text": string}, so the branch prompts can use {text}
from langchain_core.runnables import RunnableLambda

def dictionary_maker(text:str)->dict:
    return {"text":text}

dictionary_maker_runnable = RunnableLambda(dictionary_maker)

# Chapter 2.3 - Parallel chains

**Idea:** take ONE output and feed it to several chains at once with `RunnableParallel`.
Each branch runs independently and the results come back together in a dict keyed by branch name.

```
                    +-> LinkedIn chain  -+
topic -> LLM -> text|                    |-> {"LinkedIn": ..., "Instagram": ...}
                    +-> Instagram chain -+
```

### Parallel chain 1 - LinkedIn
Built as a normal `prompt | llm | parser` chain.

In [7]:
#Task-1[Prompt]
linkedin_prompt =  ChatPromptTemplate.from_messages([
    ("system","You are a linkedin post generator"),
    ("human","Create a post for the following text for LinkedIn: {text}")
])

#Task-2[LLM]
llm = ChatOpenAI(base_url="https://integrate.api.nvidia.com/v1",model="nvidia/nemotron-3.5-lightning-30b-a3b")

#Task-3[Str Parser]
str_parser = StrOutputParser()

# Finished LinkedIn chain. Expects a dict like {"text": "..."}
chain_linkedin = linkedin_prompt | llm | str_parser

## Parallel chain 2 - Instagram
Same idea, but wrapped inside a **function** + `RunnableLambda`.
This shows that a branch can be any Python code, not only a `|` chain.

In [8]:
def insta_chain(text:dict):
    # Input arrives as {"text": "..."}; pull out the string
    text=text["text"]

    #Task-1[Prompt]
    insta_prompt =  ChatPromptTemplate.from_messages([
        ("system","You are a Instagram post generator"),
        ("human","Create a post for the following text for Instagram: {text}")
    ])

    #Task-2[LLM]
    llm = ChatOpenAI(base_url="https://integrate.api.nvidia.com/v1",model="nvidia/nemotron-3.5-lightning-30b-a3b")

    #Task-3[Str Parser]
    str_parser = StrOutputParser()

    chain_instagram = insta_prompt | llm | str_parser
    # NOTE: invoking with a plain string works because the prompt has a single variable
    result=chain_instagram.invoke(text)

    return result

# Wrap so it can be used as a branch in RunnableParallel
insta_chain_runnable = RunnableLambda(insta_chain)


## Final Orchestration
Join everything: first get the text, then fan it out to both chains in parallel.

In [9]:
final_chain = (
                    prompt_template |            # 1. build prompt from the topic
                    llm |                        # 2. LLM explains the topic
                    str_parser |                 # 3. AIMessage -> string
                    dictionary_maker_runnable |  # 4. string -> {"text": string}
                    # 5. run both chains at the same time; keys become the output keys
                    RunnableParallel(branches = {"LinkedIn":chain_linkedin,"Instagram":insta_chain_runnable}) )

In [10]:
# Output shape: {'branches': {'LinkedIn': '...', 'Instagram': '...'}}
final_chain.invoke("RRR")

{'branches': {'LinkedIn': '🌟 **RRR (Rise Roar Revolt) isn’t just a film – it’s a global phenomenon.**\n\nWhen S.S. Rajamouli’s 2022 epic hit screens, it didn’t just break records in Indian cinema; it redefined what’s possible for non-English language films on the world stage. With powerhouse performances by N.T. Rama Rao Jr., Ram Charan, Ajay Devgn, and Alia Bhatt, and action sequences that set new benchmarks, RRR blended grandeur with heartfelt storytelling like never before.\n\n🎵 The Oscar-winning song “Naatu Naatu” proved that culture and rhythm can cross borders, bringing home India’s first-ever Oscar in the Best Original Song category.\n\n💡 **What we can learn:** Vision + scale + emotion = universal appeal. Whether you’re in film, business, or creativity, RRR’s journey is a masterclass in how bold storytelling resonates globally.\n\n🎬 **What’s the biggest takeaway you got from RRR?** Drop a comment!\n\n#RRR #IndianCinema #Oscars #S.SRajamouli #NaatuNaatu #GlobalStorytelling #FilmI

## Cleaning the output with another runnable
The parallel output is nested (`branches -> LinkedIn/Instagram`). Add one more `RunnableLambda`
at the end to flatten it into a simpler dict.

In [11]:
def beautify(final_response:dict)->dict:
    # Unpack the nested result and return a flat dict with lowercase keys
    linkedin_response = final_response['branches']['LinkedIn']
    instagram_response = final_response['branches']['Instagram']

    return {"linkedin":linkedin_response,"instagram":instagram_response}

beautify_runnable = RunnableLambda(beautify)

# Chains can be extended: existing chain | one more step
beautiful_chain = final_chain | beautify_runnable

# Output shape: {'linkedin': '...', 'instagram': '...'}
beautiful_chain.invoke("Pushpa")

{'linkedin': 'Here’s a polished LinkedIn post based on your text:\n\n---\n\n**Quick question: When you hear “Pushpa,” what comes to mind first? 🌸🎬**\n\nFor many, it’s the blockbuster *Pushpa: The Rise* (2021) — Allu Arjun’s pan-India sensation, directed by Sukumar, famous for the hit song *Srivalli* and the iconic line *"Jhukega Nahi"*. \n\nBut “Pushpa” (पुष्पा) means so much more. In Sanskrit, it simply translates to “flower,” and it’s a widely used name across India. Beyond the film, it’s inspired memes, music trends, and countless pop culture conversations.\n\n💡 *Fun takeaway: One word. Three meanings. A whole cultural moment.*\n\n---\n\n**Are you team Allu Arjun, a Sanskrit enthusiast, or just here for the tunes? Drop a comment below! 👇**\n\n#Pushpa #IndianCinema #Sanskrit #FilmTrivia #AlluArjun #PopCulture #TeluguHindi',
 'instagram': '🌸🔥 **Pushpa: The Movie, The Name, The Vibe!** Which one are *you* here for?\n\n🎬 **Pushpa: The Rise (2021)** – The pan-India blockbuster that broke